---
description: Shared filesystem, configuration, and progress primitives for OCR providers
output-file: preprocessing.ocr.utils.html
title: Shared OCR utilities
---

In [ ]:
# | default_exp preprocessing.ocr.utils

In [ ]:
%load_ext autoreload
%autoreload 2

These provider-neutral helpers are shared by the GLM, DashScope, and Baidu Unlimited-OCR notebooks. Keeping them in one nbdev module prevents provider implementations from drifting apart.

In [ ]:
# | export
import os
from dataclasses import dataclass
from pathlib import Path
from typing import Any


In [ ]:
# | export
def find_project_root() -> Path:
    """Find the nearest parent containing ``pyproject.toml``."""
    starts: list[Path] = []
    module_file = globals().get("__file__")
    if isinstance(module_file, str):
        starts.append(Path(module_file).resolve().parent)
    starts.append(Path.cwd().resolve())
    for start in starts:
        for candidate in [start, *start.parents]:
            if (candidate / "pyproject.toml").is_file():
                return candidate
    return Path.cwd().resolve()


def resolve_root(root_folder: Path | str) -> Path:
    """Resolve and validate an OCR input directory."""
    root = Path(root_folder).expanduser().resolve()
    if not root.exists():
        raise FileNotFoundError(f"OCR root does not exist: {root}")
    if not root.is_dir():
        raise NotADirectoryError(f"OCR root is not a directory: {root}")
    return root


def resolve_output_dir_name(output_dir_name: str) -> str:
    """Validate one relative output-directory component."""
    normalized = output_dir_name.strip()
    path = Path(normalized)
    if (
        not normalized
        or path.is_absolute()
        or len(path.parts) != 1
        or normalized in {".", ".."}
    ):
        raise ValueError("output_dir_name must be one relative directory name")
    return normalized

In [ ]:
# | export
def positive_int(value: int | None, env_name: str, default: int) -> int:
    """Resolve a positive integer from an argument or environment variable."""
    if value is None:
        configured = os.getenv(env_name, "").strip()
        if configured:
            try:
                value = int(configured)
            except ValueError as error:
                raise ValueError(f"{env_name} must be a positive integer") from error
        else:
            value = default
    if value <= 0:
        raise ValueError(f"{env_name} must be a positive integer")
    return value


def value(obj: object, name: str, default: Any = None) -> Any:
    """Read a field from either a mapping or an attribute-based response."""
    if isinstance(obj, dict):
        return obj.get(name, default)
    return getattr(obj, name, default)

In [ ]:
# | export
@dataclass
class ActivePageProgress:
    """Mutable page counters for one active OCR source."""

    source_path: Path
    pages_total: int | None = None
    pages_completed: int = 0
    last_page_elapsed_s: float | None = None


def running_in_notebook() -> bool:
    """Return whether output is being rendered by a Jupyter kernel."""
    try:
        from IPython import get_ipython
    except ImportError:
        return False
    shell = get_ipython()
    return shell is not None and shell.__class__.__name__ == "ZMQInteractiveShell"

In [ ]:
# | hide
from tempfile import TemporaryDirectory
from unittest.mock import patch

from fastcore.test import test_eq


def test_shared_ocr_utilities():
    assert (find_project_root() / "pyproject.toml").is_file()
    with TemporaryDirectory() as temporary_directory:
        root = Path(temporary_directory)
        test_eq(resolve_root(root), root.resolve())
        file_path = root / "file.txt"
        file_path.touch()
        with patch.dict(os.environ, {"OCR_TEST_CONCURRENCY": "3"}):
            test_eq(positive_int(None, "OCR_TEST_CONCURRENCY", 1), 3)
        test_eq(resolve_output_dir_name(".md"), ".md")
        test_eq(value({"status": "ok"}, "status"), "ok")
        state = ActivePageProgress(root, 2, 1, 0.25)
        test_eq((state.pages_total, state.pages_completed), (2, 1))


test_shared_ocr_utilities()